# Практика 1. S3: хранилище для данных и моделей

**Перед началом:** этот файл только для чтения и общий для всей группы.
Сохраните свою копию: **File → Save Notebook As…** →
`work/students/<ваше_имя>/01_s3.ipynb` и работайте в ней.

План:
1. Установка библиотек
2. Подключение к S3
3. Свой бакет
4. Загрузка файлов в S3
5. Просмотр содержимого
6. Скачивание файлов
7. Таблицы pandas и модели в S3
8. Копирование, удаление, ссылка для скачивания
9. **Задачи**

## 1. Установка библиотек

- **boto3** — официальный клиент S3 для Python (от Amazon, работает с любым S3-совместимым хранилищем)
- **s3fs** — даёт pandas читать и писать по адресу `s3://…`

На нашем сервере они уже стоят, команда просто это проверит. На своём компьютере она их установит.

In [ ]:
%pip install -q boto3 s3fs pandas pyarrow matplotlib

In [ ]:
import boto3
print("boto3", boto3.__version__)

## 2. Подключение к S3

Чтобы подключиться, нужны три вещи:

| Что | Где взять |
|---|---|
| адрес хранилища (`endpoint_url`) | внутри сервера `http://s3:9000`, снаружи `http://10.162.1.86:9000` |
| логин (`aws_access_key_id`) | ваш логин S3: `user1` … `user30` (выдаёт преподаватель) |
| пароль (`aws_secret_access_key`) | выдаёт преподаватель |

**Пароль никогда не пишем в код** — код попадает в Git и в чужие руки. Вводим его через `getpass`:
он спрашивается при запуске ячейки и нигде не сохраняется.

In [ ]:
import os

from getpass import getpass

S3_URL = os.environ["S3_ENDPOINT_URL"]     # http://s3:9000 — адрес S3 внутри сервера
print("адрес S3:", S3_URL)

In [ ]:
S3_USER = "user1"   # <-- ЗАМЕНИТЕ на свой логин: user1 … user30
S3_PASSWORD = os.environ.get("S3_PASSWORD") or getpass(f"Пароль S3 для {S3_USER}: ")

s3 = boto3.client("s3", endpoint_url=S3_URL,
                  aws_access_key_id=S3_USER, aws_secret_access_key=S3_PASSWORD)

# проверка связи: список бакетов — вы видите только свой
for b in s3.list_buckets()["Buckets"]:
    print(b["Name"], b["CreationDate"].date())

## 3. Свой бакет

Бакет — большой контейнер с уникальным именем. У каждого студента свой: **имя бакета = ваш логин** (`user7` → бакет `user7`).

Бакет уже создан для вас заранее. Права настроены так, что вы видите и можете делать всё **только в своём бакете**.
Создать бакет с другим именем или заглянуть в чужой нельзя — сервер ответит `AccessDenied`. Это и есть разграничение доступа в S3.

In [ ]:
BUCKET = S3_USER
print("мой бакет:", BUCKET)

In [ ]:
from botocore.exceptions import ClientError

def bucket_exists(name: str) -> bool:
    try:
        s3.head_bucket(Bucket=name)   # head_* — узнать, есть ли объект, не скачивая его
        return True
    except ClientError:
        return False

if not bucket_exists(BUCKET):
    s3.create_bucket(Bucket=BUCKET)
    print("создан бакет", BUCKET)
else:
    print("бакет уже есть:", BUCKET)

In [ ]:
# попробуем заглянуть в чужой бакет — должно быть запрещено
try:
    s3.list_objects_v2(Bucket="students")
    print("доступ есть?!")
except ClientError as e:
    print("чужой бакет:", e.response["Error"]["Code"])

## 4. Загрузка файлов в S3

Три способа:

| Метод | Когда |
|---|---|
| `upload_file(путь, бакет, ключ)` | файл уже лежит на диске |
| `put_object(Bucket, Key, Body)` | данные в памяти: строка или байты |
| `upload_fileobj(файл, бакет, ключ)` | открытый файл или поток |

**Ключ** — полное имя объекта. Слеши в ключе выглядят как папки, но это просто часть имени.

In [ ]:
# 4.1 файл с диска
with open("hello.txt", "w") as f:
    f.write("Привет, S3!\n")

s3.upload_file("hello.txt", BUCKET, "docs/hello.txt")
print("загружен docs/hello.txt")

In [ ]:
# 4.2 строка из памяти, без файла на диске
s3.put_object(Bucket=BUCKET, Key="docs/note.txt", Body="Заметка из памяти".encode("utf-8"))
print("загружен docs/note.txt")

In [ ]:
# 4.3 таблица pandas -> CSV в S3
import pandas as pd
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True).frame
s3.put_object(Bucket=BUCKET, Key="data/raw/wine.csv", Body=wine.to_csv(index=False))
print("загружен data/raw/wine.csv:", wine.shape)

In [ ]:
# 4.4 картинка matplotlib -> PNG в S3 (через память, без диска)
import io
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
wine["alcohol"].hist(ax=ax, bins=20)
ax.set_title("Алкоголь в винах")

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=100)
plt.close(fig)
s3.put_object(Bucket=BUCKET, Key="images/alcohol.png", Body=buf.getvalue(), ContentType="image/png")
print("загружен images/alcohol.png,", len(buf.getvalue()), "байт")

## 5. Что лежит в бакете

`list_objects_v2` возвращает до 1000 объектов. `Prefix` — показать только ключи, начинающиеся с этой строки («папка»).

In [ ]:
def ls(prefix: str = ""):
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    for obj in resp.get("Contents", []):
        print(f'{obj["Key"]:<30} {obj["Size"]:>8} байт   {obj["LastModified"]:%Y-%m-%d %H:%M}')

ls()

In [ ]:
ls("docs/")   # только «папка» docs

In [ ]:
# метаданные одного объекта, без скачивания
meta = s3.head_object(Bucket=BUCKET, Key="images/alcohol.png")
print(meta["ContentLength"], "байт,", meta["ContentType"])

## 6. Скачивание

| Метод | Результат |
|---|---|
| `download_file(бакет, ключ, путь)` | файл на диске |
| `get_object(Bucket, Key)["Body"].read()` | байты в памяти |

In [ ]:
# 6.1 на диск
s3.download_file(BUCKET, "docs/hello.txt", "hello_downloaded.txt")
print(open("hello_downloaded.txt").read())

In [ ]:
# 6.2 в память
body = s3.get_object(Bucket=BUCKET, Key="docs/note.txt")["Body"].read()
print(body.decode("utf-8"))

In [ ]:
# 6.3 CSV из S3 сразу в pandas
body = s3.get_object(Bucket=BUCKET, Key="data/raw/wine.csv")["Body"].read()
df = pd.read_csv(io.BytesIO(body))
df.head()

In [ ]:
# 6.4 картинку из S3 — показать в ноутбуке
from IPython.display import Image
Image(s3.get_object(Bucket=BUCKET, Key="images/alcohol.png")["Body"].read())

## 7. Parquet и модели в S3

### Зачем это нужно
В ML-конвейере данные сохраняют **много раз**: сырые → очищенные → train/test → прогнозы.
Если каждый раз сохранять в CSV, возникают три проблемы:

1. **Размер.** CSV — это текст: число `300.25` занимает 6 байт, в parquet — меньше и ещё сжимается.
2. **Скорость.** CSV нужно разбирать построчно, parquet читается почти сразу.
3. **Типы теряются.** Сохранили дату в CSV — прочитали обратно **строку**. Parquet помнит, что это дата.

**Parquet** — формат таблиц, который решает все три проблемы. Поэтому в ML его используют почти везде
(в наших DAG-ах train/test тоже лежат в `.parquet`).

### Что сделаем
- **7.1** — сравним CSV и parquet на таблице продаж в **миллион строк** (как в примере с магазином из лекции);
- **7.2** — научимся класть таблицу в S3 в parquet и читать обратно: это приём, который вы будете использовать в Airflow;
- **7.3** — так же сохраним в S3 **модель**.

In [ ]:
# 7.1 CSV против parquet на большой таблице (всё в памяти, в S3 пока не пишем)
import time
import numpy as np

n = 1_000_000
rng = np.random.default_rng(42)
sales = pd.DataFrame({
    "time": pd.date_range("2026-01-01", periods=n, freq="min"),   # дата и время продажи
    "store": rng.choice(["Москва", "Казань", "Сочи"], n),
    "product_id": rng.integers(1, 5000, n),
    "price": rng.normal(300, 50, n).round(2),
    "sold": rng.integers(0, 20, n),
})

csv_bytes = sales.to_csv(index=False).encode()        # таблица -> CSV (байты)
pq_buf = io.BytesIO()
sales.to_parquet(pq_buf, index=False)                  # таблица -> parquet (байты)
pq_bytes = pq_buf.getvalue()

t = time.time(); from_csv = pd.read_csv(io.BytesIO(csv_bytes)); t_csv = time.time() - t
t = time.time(); from_pq = pd.read_parquet(io.BytesIO(pq_bytes)); t_pq = time.time() - t

print(f"Размер:  CSV {len(csv_bytes)/1e6:.1f} МБ   parquet {len(pq_bytes)/1e6:.1f} МБ   (в {len(csv_bytes)/len(pq_bytes):.1f} раза меньше)")
print(f"Чтение:  CSV {t_csv:.2f} с     parquet {t_pq:.2f} с")
print("Тип колонки time после CSV:    ", from_csv["time"].dtype, " <- строка, дату надо разбирать заново")
print("Тип колонки time после parquet:", from_pq["time"].dtype)

**Что видно:** parquet в несколько раз меньше, читается во много раз быстрее, и дата осталась датой.
Для ML это значит: меньше места в S3, быстрее задачи Airflow и нет ошибок из-за «испорченных» типов.

Ещё один плюс — можно прочитать **только нужные колонки**, не загружая всю таблицу:

In [ ]:
t = time.time()
only_sold = pd.read_parquet(io.BytesIO(pq_bytes), columns=["sold"])
print(f"одна колонка из миллиона строк: {time.time() - t:.3f} с")

> На маленькой таблице (150–200 строк, как Iris или Wine) parquet может оказаться даже **больше** CSV:
> в файле есть служебная информация о колонках. Выигрыш появляется на больших данных — а в реальных задачах они большие.

### 7.2 Таблица в S3 в формате parquet и обратно
Схема всегда одна и та же, её стоит запомнить:

**сохранить:** DataFrame → `to_parquet` в буфер памяти → `put_object`<br>
**прочитать:** `get_object` → байты → `read_parquet`

In [ ]:
# сохранить: DataFrame -> parquet в памяти -> S3
buf = io.BytesIO()
sales.to_parquet(buf, index=False)
s3.put_object(Bucket=BUCKET, Key="data/processed/sales.parquet", Body=buf.getvalue())

# прочитать: S3 -> байты -> DataFrame
body = s3.get_object(Bucket=BUCKET, Key="data/processed/sales.parquet")["Body"].read()
sales_back = pd.read_parquet(io.BytesIO(body))

print("в S3:", s3.head_object(Bucket=BUCKET, Key="data/processed/sales.parquet")["ContentLength"] // 1_000_000, "МБ")
print("прочитали", sales_back.shape, "| совпадает с исходной:", sales_back.equals(sales))

### 7.3 Модель в S3
Модель sklearn — это объект Python, а S3 хранит байты. `joblib.dump` превращает модель в байты,
`joblib.load` — обратно. Так модель, обученную в одной задаче Airflow, загружает следующая задача
или сервис прогнозов на другом компьютере.

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression

X, y = wine.drop(columns="target"), wine["target"]
model = LogisticRegression(max_iter=5000).fit(X, y)

buf = io.BytesIO()
joblib.dump(model, buf)                                         # модель -> байты
s3.put_object(Bucket=BUCKET, Key="models/logreg.joblib", Body=buf.getvalue())

body = s3.get_object(Bucket=BUCKET, Key="models/logreg.joblib")["Body"].read()
loaded = joblib.load(io.BytesIO(body))                          # байты -> модель
print("прогноз загруженной модели:", loaded.predict(X.head(5)))

## 8. Копирование, удаление, ссылка для скачивания

In [ ]:
# 8.1 копия внутри S3 (данные не проходят через ваш компьютер)
s3.copy_object(Bucket=BUCKET, Key="backup/wine.csv",
               CopySource={"Bucket": BUCKET, "Key": "data/raw/wine.csv"})
ls("backup/")

In [ ]:
# 8.2 удаление
s3.delete_object(Bucket=BUCKET, Key="backup/wine.csv")
ls("backup/")
print("после удаления в backup/ пусто")

In [ ]:
# 8.3 временная ссылка на файл (presigned URL): открывается в браузере без пароля, пока не истечёт
public = boto3.client("s3", endpoint_url="http://10.162.1.86:9000",   # адрес, видимый снаружи
                      aws_access_key_id=S3_USER, aws_secret_access_key=S3_PASSWORD)
url = public.generate_presigned_url("get_object",
                                    Params={"Bucket": BUCKET, "Key": "images/alcohol.png"},
                                    ExpiresIn=600)   # 10 минут
print(url)

## Шпаргалка

| Действие | Команда |
|---|---|
| подключиться | `s3 = boto3.client("s3", endpoint_url=..., aws_access_key_id=..., aws_secret_access_key=...)` |
| бакеты | `s3.list_buckets()`, `s3.create_bucket(Bucket=b)` |
| загрузить | `s3.upload_file(путь, b, key)`, `s3.put_object(Bucket=b, Key=key, Body=...)` |
| список | `s3.list_objects_v2(Bucket=b, Prefix="docs/")` |
| скачать | `s3.download_file(b, key, путь)`, `s3.get_object(Bucket=b, Key=key)["Body"].read()` |
| метаданные | `s3.head_object(Bucket=b, Key=key)` |
| копировать / удалить | `s3.copy_object(...)`, `s3.delete_object(Bucket=b, Key=key)` |
| ссылка | `s3.generate_presigned_url("get_object", Params=..., ExpiresIn=600)` |

---
# 9. Задачи

Все задачи — в **своём бакете** `BUCKET`. Проверяйте результат через `ls()` и в веб-консоли S3.

### Задача 1. Iris в S3
Загрузите датасет Iris (`sklearn.datasets.load_iris(as_frame=True).frame`) в свой бакет как CSV
по ключу `data/raw/iris.csv`. Выведите содержимое «папки» `data/raw/`.

In [ ]:
# ваш код здесь

### Задача 2. Несколько файлов в цикле
Создайте 3 текстовых отчёта `reports/report_1.txt` … `reports/report_3.txt`
(в каждом — строка «Отчёт номер N»). Потом одной командой получите список `reports/`
и посчитайте **общий размер** этих файлов в байтах.

In [ ]:
# ваш код здесь

### Задача 3. Скачать и проверить
Скачайте `data/raw/iris.csv` **на диск** в файл `iris_local.csv`, прочитайте его pandas
и проверьте, что строк столько же, сколько в исходном датасете (150).

In [ ]:
# ваш код здесь

### Задача 4. Parquet туда и обратно
Сохраните Iris в S3 в формате **parquet** по ключу `data/processed/iris.parquet`,
прочитайте обратно в DataFrame и проверьте `.equals()`. Сравните размеры CSV и parquet (`head_object`).
Какой файл меньше и почему? Что изменится для таблицы в миллион строк?

In [ ]:
# ваш код здесь

### Задача 5. График и ссылка
Постройте диаграмму рассеяния Iris (`petal length (cm)` против `petal width (cm)`, цвет — `target`),
сохраните её в S3 как `images/iris_scatter.png` **без записи на диск** и получите ссылку на 10 минут.
Откройте ссылку в браузере.

In [ ]:
# ваш код здесь

### Задача 6. Модель в S3
Обучите `RandomForestClassifier(random_state=42)` на Iris, сохраните модель в `models/iris_rf.joblib`,
загрузите её обратно и посчитайте accuracy загруженной модели на всём датасете.

In [ ]:
# ваш код здесь

### Задача 7 (со звёздочкой). Резервная копия «папки»
Напишите функцию `backup(prefix)`, которая копирует **все** объекты с ключами на `prefix`
в `backup/<prefix>…` через `copy_object`. Сделайте резервную копию `reports/`,
затем удалите оригиналы из `reports/` и покажите, что копии на месте.

In [ ]:
# ваш код здесь

---
**Готово?** Откройте веб-консоль S3 (`http://10.162.1.86:9001/rustfs/console/index.html`),
войдите **своим логином и паролем S3** и посмотрите файлы своего бакета глазами. Дальше — Airflow: те же операции, но через `S3Hook`
и по расписанию.